In [1]:
# Force install by calling the files directly and ignoring compatibility tags
!pip install /kaggle/input/datasets/pjleek/onnxruntime126-312/onnxruntime-1.26.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl --no-deps --force-reinstall


Processing /kaggle/input/datasets/pjleek/onnxruntime126-312/onnxruntime-1.26.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl


In [2]:
!pip install onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 10.0 MB/s eta 0:00:00


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import json
import glob
import math
import numpy as np
import os
from collections import namedtuple

Planet = namedtuple("Planet",["id", "owner", "x", "y", "radius", "ships", "production"])
CENTER_X, CENTER_Y = 50.0, 50.0
MAX_SPEED = 6.0

# ============================================================
# 1. The Value-Estimator Neural Network (The Brain)
# ============================================================
class OrbitNet(nn.Module):
    def __init__(self):
        super(OrbitNet, self).__init__()
        self.fc1 = nn.Linear(10, 64)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(64, 32)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid() # Keeps outputs bounded between 0.0 (Bench) and 1.0 (MVP)

    def forward(self, x):
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        return self.sigmoid(self.fc3(x))

# ============================================================
# 2. Physics & Future State Parsing
# ============================================================
def fleet_speed(ships):
    if ships <= 1: return 1.0
    return 1.0 + (MAX_SPEED - 1.0) * (min(1.0, math.log(ships) / math.log(1000.0)) ** 1.5)

def get_target_pos(tgt, turns, ang_vel, initial_planets, comets, comet_ids):
    if tgt.id in comet_ids:
        for c in comets:
            if tgt.id in c.get("planet_ids", []):
                idx = c["planet_ids"].index(tgt.id)
                f_idx = c.get("path_index", 0) + turns
                if idx < len(c["paths"]) and 0 <= f_idx < len(c["paths"][idx]): return c["paths"][idx][f_idx]
        return None
    init = initial_planets.get(tgt.id)
    if not init: return (tgt.x, tgt.y)
    r = math.hypot(init.x - CENTER_X, init.y - CENTER_Y)
    if r + init.radius >= 50.0: return (tgt.x, tgt.y)
    ang = math.atan2(tgt.y - CENTER_Y, tgt.x - CENTER_X) + ang_vel * turns
    return (CENTER_X + r * math.cos(ang), CENTER_Y + r * math.sin(ang))

def plan_flight(src, tgt, ships, ang_vel, initial_planets, comets, comet_ids):
    speed = fleet_speed(ships)
    tx, ty = tgt.x, tgt.y
    eta = 0.0
    for _ in range(5): 
        flight_dist = max(0.0, math.hypot(tx - src.x, ty - src.y) - src.radius - 0.1 - tgt.radius)
        eta = flight_dist / speed
        pos = get_target_pos(tgt, int(math.ceil(eta)), ang_vel, initial_planets, comets, comet_ids)
        if not pos: return None, 999
        tx, ty = pos
    return int(math.ceil(eta)), tx, ty, speed

# ============================================================
# 3. The Coach's Rubric (Composite Reward Shaping)
# ============================================================
def calculate_coach_score(is_winner, tgt_prod, speed, tactical_success):
    """
    Grades every single launch based on the DNA of our Playbook.
    Final_Score = (0.4 * Win/Loss) + (0.3 * Prod) + (0.2 * Speed) + (0.1 * Precision)
    """
    win_score = 0.4 if is_winner else 0.0
    prod_score = 0.3 * (tgt_prod / 5.0)
    speed_score = 0.2 * min(1.0, speed / 6.0)
    tactical_score = 0.1 if tactical_success else 0.0
    
    final_score = win_score + prod_score + speed_score + tactical_score
    
    # THE SACK PENALTY: 
    if not tactical_success:
        if is_winner:
            # The winner made a blunder but got away with it.
            final_score *= 0.5 
        else:
            # The loser made a fatal error. Hard negative.
            final_score *= 0.1 
            
    return final_score

# ============================================================
# 4. Data Extraction
# ============================================================
def extract_training_data(dataset_path, max_files=300):
    files = glob.glob(os.path.join(dataset_path, "**/*.json"), recursive=True)
    print(f"Found {len(files)} replays. Building Coach's Playbook from {max_files}...")
    
    X_data, Y_data = [], []
    
    for filepath in files[:max_files]:
        try:
            with open(filepath, 'r') as f: match = json.load(f)
            steps = match.get("steps",[])
            if not steps: continue
            
            # Identify Winners and Losers
            last_step = steps[-1]
            rewards =[agent.get("reward", -1) for agent in last_step]
            winner_id = rewards.index(max(rewards))
            
            for step_idx, step in enumerate(steps[:-1]):
                obs = step[0].get("observation", {})
                raw_planets = obs.get("planets",[])
                if not raw_planets: continue
                
                planets =[Planet(*p) for p in raw_planets]
                ang_vel = obs.get("angular_velocity", 0.0)
                initial_planets = {Planet(*p).id: Planet(*p) for p in obs.get("initial_planets", [])}
                comets = obs.get("comets",[])
                comet_ids = set(obs.get("comet_planet_ids",[]))
                
                max_prod = max([p.production for p in planets]) if planets else 1
                is_4p = 1.0 if len(set([p.owner for p in planets if p.owner != -1])) > 2 else 0.0
                
                # Analyze EVERY player's moves (Winner & Losers)
                for player_id in range(len(step)):
                    action_list = step[player_id].get("action",[])
                    is_winner = (player_id == winner_id)
                    
                    for action in action_list:
                        src_id, angle, ships_sent = action[0], action[1], max(1, action[2])
                        src_p = next((p for p in planets if p.id == src_id), None)
                        if not src_p: continue
                        
                        # Reverse-engineer the target
                        best_diff, tgt_p = float('inf'), None
                        for p in planets:
                            if p.id == src_id: continue
                            tgt_ang = math.atan2(p.y - src_p.y, p.x - src_p.x)
                            ang_diff = abs((angle - tgt_ang + math.pi) % (2 * math.pi) - math.pi)
                            if ang_diff < best_diff:
                                best_diff, tgt_p = ang_diff, p
                                
                        if best_diff > 0.2 or not tgt_p: continue # Unclear target, skip
                            
                        # Physics Sync
                        res = plan_flight(src_p, tgt_p, ships_sent, ang_vel, initial_planets, comets, comet_ids)
                        if not res: continue
                        eta, tx, ty, speed = res
                        if eta == 999: continue
                        
                        # Look into the future to grade Tactical Precision
                        # Give it a 2-turn buffer to resolve combat
                        future_turn = min(len(steps) - 1, step_idx + eta + 2)
                        future_obs = steps[future_turn][0].get("observation", {}).get("planets",[])
                        future_tgt = next((p for p in future_obs if p[0] == tgt_p.id), None)
                        
                        tactical_success = (future_tgt is not None and future_tgt[1] == player_id)
                        
                        # Apply the Coach's Rubric
                        composite_score = calculate_coach_score(is_winner, tgt_p.production, speed, tactical_success)
                        
                        # 10-Feature Vector
                        npv = (tgt_p.production * max(0, 500 - step_idx - eta)) / 1000.0
                        capital_risk = ships_sent / max(1.0, float(src_p.ships))
                        time_discount = eta / 50.0
                        raw_yield = tgt_p.production / 5.0
                        ownership = 1.0 if tgt_p.owner != -1 else 0.5
                        in_quad = 1.0 if (1 if src_p.x>50 else -1) == (1 if tgt_p.x>50 else -1) and (1 if src_p.y>50 else -1) == (1 if tgt_p.y>50 else -1) else 0.0
                        norm_speed = speed / 6.0 
                        displacement = math.hypot(tgt_p.x - tx, tgt_p.y - ty) / 100.0
                        
                        feat =[npv, capital_risk, time_discount, raw_yield, ownership, in_quad, step_idx / 500.0, is_4p, norm_speed, displacement]
                        
                        X_data.append(feat)
                        Y_data.append([composite_score]) # THE NEW CONTINUOUS LABEL
                        
                        # Add a "Read the Defense" Negative Sample for Winners
                        if is_winner:
                            bad_tgt = planets[np.random.randint(len(planets))]
                            if bad_tgt.id not in (src_id, tgt_p.id):
                                hyp_ships = int(math.ceil((bad_tgt.ships + 1) * 1.1))
                                hyp_res = plan_flight(src_p, bad_tgt, hyp_ships, ang_vel, initial_planets, comets, comet_ids)
                                if hyp_res:
                                    h_eta, h_tx, h_ty, h_speed = hyp_res
                                    h_npv = (bad_tgt.production * max(0, 500 - step_idx - h_eta)) / 1000.0
                                    h_risk = hyp_ships / max(1.0, float(src_p.ships))
                                    h_in_quad = 1.0 if (1 if src_p.x>50 else -1) == (1 if bad_tgt.x>50 else -1) and (1 if src_p.y>50 else -1) == (1 if bad_tgt.y>50 else -1) else 0.0
                                    h_feat =[h_npv, h_risk, h_eta/50.0, bad_tgt.production/5.0, 1.0 if bad_tgt.owner != -1 else 0.5, h_in_quad, step_idx/500.0, is_4p, h_speed/6.0, math.hypot(bad_tgt.x-h_tx, bad_tgt.y-h_ty)/100.0]
                                    
                                    X_data.append(h_feat)
                                    # Penalize heavily for a random target the winner ignored
                                    Y_data.append([0.05]) 
                            
        except Exception as e:
            continue
            
    return torch.tensor(X_data, dtype=torch.float32), torch.tensor(Y_data, dtype=torch.float32)

# ============================================================
# 5. Model Training & Safe ONNX Export
# ============================================================
def train_and_export():
    dataset_path = "/kaggle/input/datasets/bovard/orbit-wars-top10-episodes-2026-05-04/episodes/episodes"
    X_train, Y_train = extract_training_data(dataset_path, max_files=300)
    
    if len(X_train) == 0:
        print("No training data found! Check the file path.")
        return
        
    print(f"Extracted {len(X_train)} training scenarios. Forging the Brain...")
    
    model = OrbitNet()
    optimizer = optim.Adam(model.parameters(), lr=0.005)
    
    # SWITCHED TO MSE: We are now predicting a continuous Value Score, not a binary True/False
    criterion = nn.MSELoss() 
    
    for epoch in range(150): 
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, Y_train)
        loss.backward()
        optimizer.step()
        if epoch % 25 == 0: print(f"Epoch {epoch} - Mean Squared Error Loss: {loss.item():.4f}")
        
    print("Exporting to ONNX...")
    model.eval() 
    dummy_input = torch.randn(1, 10)
    export_path = "/kaggle/working/orbit_model.onnx"
    
    import onnx
    torch.onnx.export(
        model, dummy_input, export_path, export_params=True,
        opset_version=15, do_constant_folding=True,
        input_names=['input'], output_names=['output']
    )
    
    # Force IR Version 9 for Kaggle backend compatibility
    onnx_model = onnx.load(export_path)
    onnx_model.ir_version = 9
    onnx.save(onnx_model, export_path)
    
    print(f"Successfully saved Reward-Shaped IR-9 Model to: {export_path}")

if __name__ == "__main__":
    train_and_export()

Found 2631 replays. Building Coach's Playbook from 300...
Extracted 35859 training scenarios. Forging the Brain...
Epoch 0 - Mean Squared Error Loss: 0.1210
Epoch 25 - Mean Squared Error Loss: 0.0556
Epoch 50 - Mean Squared Error Loss: 0.0458
Epoch 75 - Mean Squared Error Loss: 0.0448
Epoch 100 - Mean Squared Error Loss: 0.0443
Epoch 125 - Mean Squared Error Loss: 0.0440
Exporting to ONNX...


W0510 04:52:56.882000 23 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 15 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `OrbitNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `OrbitNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 15).


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Successfully saved Reward-Shaped IR-9 Model to: /kaggle/working/orbit_model.onnx
